# 
理解事件驱动的agent设计，离不开对异步编程知识的深入理解，本篇主要是以python为例，回顾一下异步编程的基本知识。
异步编程，本质上是利用现代CPU的多线程处理能力，让任务并行的跑在不同的CPU核心上，从而加速程序的执行；python在3.14版本以前，由于GIL的限制，实际上是不能利用CPU多核执行任务的(除非使用多进程)，但是仍然有许多io耗时的任务，可以通过异步执行的方式来进行加速(这得益于python底层依赖的库的并行执行机制)，也就是python中的协程。

In [ ]:
import asyncio
async def my_coro():
    print(f'begin')
    await asyncio.sleep(1)
    print(f'end')

await my_coro() #在jupyter notebook里可以直接await，是因为jupyter notebook已经是在event loop里，在真实的项目里应该用 asyncio.run(my_coro)来执行

begin
end


In [3]:
task = asyncio.create_task(my_coro())
await asyncio.gather(task)

begin
end


[None]

In [ ]:
from concurrent.futures import ThreadPoolExecutor,ProcessPoolExecutor,Future,as_completed
from functools import partial
import time

class MyPool:
    def __init__(self,worker_nums: int = 1) -> None:
        self._threadPool = ThreadPoolExecutor(max_workers=worker_nums)
        self._processPool = ProcessPoolExecutor(max_workers=worker_nums)

    def run_task(self, fn):
        future: Future = self._threadPool.submit(fn)
        result = future.result()
        return result

    def run_cpu_parallel(self,fns: list):
       
       futures = [self._processPool.submit(fn) for fn in fns]
       results = []
       for fut in as_completed(futures):
           results.append(fut.result())
       return results
        

def time_consume_work():
    time.sleep(2)
    return 'done'

def cpu_consume_work(des: str):
    i = 0
    for _ in range(10000):
        i += 1
    print(f'i = {i}')
    return f'{des} finished..'


pool = MyPool(2)
pool.run_task(time_consume_work)

pool.run_cpu_parallel([partial(cpu_consume_work,'work1'),partial(cpu_consume_work,'work2')])




i = 10000
i = 10000


TypeError: 'str' object is not callable